# BankChurners.csv — Individual Preprocessing Contribution
**Student ID:** IT25102086

**Assigned Technique:** Encoding Categorical Variables (Label Encoding)

**Project:** Predicting Credit Card Customer Churn Using Multi-Quarter Usage Trends

**Instructions for Google Colab:**
1. Run the first cell below to upload `BankChurners.csv` from your computer.
2. Then run each cell in order (Shift + Enter).

## Step 0: Upload the Dataset (Google Colab)

In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose BankChurners.csv when prompted


## Step 1: Load Dataset & Identify Categorical Columns

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np

df = pd.read_csv("BankChurners.csv")

# Drop the ID column and the bundled leakage columns so they don't interfere
leakage_cols = [c for c in df.columns if c.startswith("Naive_Bayes_Classifier")]
df = df.drop(columns=["CLIENTNUM"] + leakage_cols)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns found:", categorical_cols)
df[categorical_cols].head()


**Why:** Machine learning algorithms (Logistic Regression, KNN, SVM, tree-based
models, etc.) work on numbers, not text. Before we can feed this dataset into
any model, every text column has to be converted into a numeric form. The
columns identified here — `Attrition_Flag` (target), `Gender`,
`Education_Level`, `Marital_Status`, `Income_Category`, and `Card_Category` —
are all stored as strings and must be encoded.

## Step 2: Inspect Category Values Before Encoding

In [ ]:
for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")


**Why:** Checking the unique values first shows us what each column actually
contains — for example `Marital_Status` and `Income_Category` include an
`'Unknown'` category, which Label Encoding will simply treat as its own
class rather than a missing value. Confirming this before encoding avoids
surprises later.

## Step 3: Apply Label Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"{col} mapping: {mapping}")

df.head()


**Why:** `LabelEncoder` assigns each unique category an integer code (e.g.
`Existing Customer` → 1, `Attrited Customer` → 0). This is the technique
assigned for this task, so it is applied consistently across every
categorical column, including the target `Attrition_Flag`. A separate
encoder is kept for each column (in `label_encoders`) so the same mapping
can be reversed or reused later if needed — for example, to turn predictions
back into readable labels.

**Note on ordinal risk:** Label Encoding works cleanly for binary columns
like `Gender` and `Attrition_Flag`, and is acceptable here for the
multi-category columns since many tree-based models (Random Forest, XGBoost)
handle integer-coded categories without assuming a false order. If a
distance-based model is used later, this is a trade-off worth mentioning in
the viva: One-Hot Encoding would avoid implying an order between categories
such as `Education_Level`, but Label Encoding was the technique assigned for
this contribution.

## Step 4: Verify the Encoded Dataset

In [ ]:
print("Data types after encoding:\n")
print(df.dtypes)
print("\nAny remaining non-numeric columns:", 
      df.select_dtypes(include=['object']).columns.tolist())


**Why:** This final check confirms every column is now numeric. An empty
list here means the dataset is fully ready to be combined with the rest of
the group's pipeline (missing value handling, outlier detection, correlation
analysis, scaling, and the train/test split) without any text column causing
an error further down the line.

## Step 5: EDA Visualization — Churn Rate by Encoded Category

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x=df['Gender'], hue=df['Attrition_Flag'], ax=axes[0])
axes[0].set_title('Attrition by Gender (encoded)')
axes[0].set_xlabel('Gender (0/1)')

sns.countplot(x=df['Education_Level'], hue=df['Attrition_Flag'], ax=axes[1])
axes[1].set_title('Attrition by Education Level (encoded)')
axes[1].set_xlabel('Education Level (encoded)')

plt.tight_layout()
plt.show()


**Interpretation:** With the categorical columns now numeric, they can be
plotted and compared against the target directly. The counts show that
attrition (`Attrition_Flag` = 0, i.e. Attrited Customer) is spread across all
genders and education levels rather than being concentrated in one category,
suggesting these fields alone are weak predictors of churn on their own — but
they still add useful signal when combined with the numeric usage features
the rest of the group is preparing.

## Summary

| Column | Technique | Values Encoded |
|---|---|---|
| Attrition_Flag (target) | Label Encoding | Existing Customer, Attrited Customer |
| Gender | Label Encoding | M, F |
| Education_Level | Label Encoding | High School, Graduate, Uneducated, Unknown, College, Post-Graduate, Doctorate |
| Marital_Status | Label Encoding | Married, Single, Unknown, Divorced |
| Income_Category | Label Encoding | $60K-$80K, Less than $40K, $80K-$120K, $40K-$60K, $120K+, Unknown |
| Card_Category | Label Encoding | Blue, Gold, Silver, Platinum |

This notebook is my individual contribution and feeds directly into the
group's `group_pipeline.ipynb`, where it is combined with missing value
handling, outlier detection, correlation analysis, feature scaling, and the
train/test split.